This notebook documents the fine-tuning run of `facebook/nllb-200-distilled-600M`
on 137,418 English–Sesotho medical sentence pairs. It covers the full three-epoch
training process — including the epoch 3 resume from the epoch 2 checkpoint
(val=1.7153) — and exports the final best model (val=1.6941) for use in evaluation.


In [ ]:
import os
print(f"Current PID: {os.getpid()}")


Current PID: 3789


In [7]:
import torch, os
print(f"PID: {os.getpid()}")
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"GPU used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


PID: 3789
GPU free: 0.60 GB
GPU used: 13.76 GB


In [ ]:
import subprocess, torch, os

print(f"Current PID: {os.getpid()}")
print(f"GPU free:    {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"GPU used:    {torch.cuda.memory_allocated()/1e9:.2f} GB\n")

# Show all processes holding the GPU
result = subprocess.run(
    ['nvidia-smi', '--query-compute-apps=pid,used_memory,process_name',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
print("Processes on GPU:")
print(result.stdout if result.stdout else "  (none)")


Current PID: 3789
GPU free:    0.60 GB
GPU used:    13.76 GB

Processes on GPU:
3789, 14336 MiB, /home/zeus/miniconda3/envs/cloudspace/bin/python



In [11]:
import pandas as pd

df = pd.read_csv("/teamspace/studios/this_studio/full_corpus.csv")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(2))


Rows: 137,418
Columns: ['english', 'sesotho', 'source']
                                             english  \
0  Antiretroviral therapy should be started as so...   
1  All people living with HIV should receive anti...   

                                             sesotho            source  
0  Phekolo ea lithibela-mafofore (Antiretroviral ...  verified_medical  
1  Batho bohle ba phelang le tšoaetso ea HIV ba l...  verified_medical  


In [1]:
import torch
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.1f} GB / {total/1e9:.1f} GB")


GPU free: 15.5 GB / 15.6 GB


In [ ]:
import os, gc, time
import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSeq2SeqLM, NllbTokenizerFast
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

#  Config 
MODEL_NAME  = "facebook/nllb-200-distilled-600M"
SRC_LANG    = "eng_Latn"
TGT_LANG    = "sot_Latn"
MAX_LEN     = 96
BATCH_SIZE  = 2
ACCUM_STEPS = 32
EPOCHS      = 3          # total epochs the scheduler was built for
RESUME_FROM = 2          # ← skip epochs 1 & 2, start at epoch 3
LR          = 2e-5
SAVE_DIR    = "/teamspace/studios/this_studio/checkpoints"
DATA_PATH   = "/teamspace/studios/this_studio/full_corpus.csv"
Path(SAVE_DIR).mkdir(exist_ok=True)

#  Load & clean data 
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["english", "sesotho"])
df = df[df["english"].str.strip().str.len() > 0]
df = df[df["sesotho"].str.strip().str.len() > 0]
df = df.reset_index(drop=True)
print(f"Clean rows: {len(df):,}")

train_df, val_df = train_test_split(df, test_size=0.05, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}")

#  Tokenizer — load from saved checkpoint 
tokenizer = NllbTokenizerFast.from_pretrained(f"{SAVE_DIR}/best_model")
tokenizer.src_lang = SRC_LANG
print("Tokenizer loaded ")

# ─ Model — load from saved checkpoint 
model = AutoModelForSeq2SeqLM.from_pretrained(
    f"{SAVE_DIR}/best_model",    
    torch_dtype=torch.float32,
)
model = model.cuda()
model.gradient_checkpointing_enable()
free, _ = torch.cuda.mem_get_info()
print(f"Model loaded  — GPU free: {free/1e9:.2f} GB")

#  Dataset 
class TranslationDataset(Dataset):
    def __init__(self, dataframe):
        self.src = dataframe["english"].tolist()
        self.tgt = dataframe["sesotho"].tolist()
    def __len__(self):
        return len(self.src)
    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]

def collate_fn(batch):
    src_texts, tgt_texts = zip(*batch)
    tokenizer.src_lang = SRC_LANG
    src_enc = tokenizer(list(src_texts), max_length=MAX_LEN,
                        padding=True, truncation=True, return_tensors="pt")
    tokenizer.src_lang = TGT_LANG
    tgt_enc = tokenizer(list(tgt_texts), max_length=MAX_LEN,
                        padding=True, truncation=True, return_tensors="pt")
    tokenizer.src_lang = SRC_LANG
    labels = tgt_enc["input_ids"].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    return {"input_ids":      src_enc["input_ids"],
            "attention_mask": src_enc["attention_mask"],
            "labels":         labels}

train_loader = DataLoader(TranslationDataset(train_df), batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(TranslationDataset(val_df),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, collate_fn=collate_fn)

#  Sanity check 
sample = next(iter(train_loader))
model.train()
scaler = torch.cuda.amp.GradScaler()

with torch.autocast(device_type="cuda", dtype=torch.float16):
    out = model(input_ids=sample["input_ids"].cuda(),
                attention_mask=sample["attention_mask"].cuda(),
                labels=sample["labels"].cuda())
loss_val = out.loss.item()
print(f"\nSanity loss: {loss_val:.4f}  ← must not be nan")
assert loss_val == loss_val, "NaN loss on sanity — stop here"
del out, sample
torch.cuda.empty_cache()
free, _ = torch.cuda.mem_get_info()
print(f"Post-sanity GPU free: {free/1e9:.2f} GB")
print(" Sanity passed — resuming from epoch 3\n")

#  Optimizer & Scheduler 
# Rebuild scheduler for full 3-epoch budget, then fast-forward
optimizer   = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = total_steps // 10
scheduler   = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

# Fast-forward scheduler to where epoch 3 begins
steps_per_epoch = len(train_loader) // ACCUM_STEPS
completed_steps = steps_per_epoch * RESUME_FROM
for _ in range(completed_steps):
    scheduler.step()
print(f"Scheduler fast-forwarded {completed_steps} steps ✅")
print(f"Train batches: {len(train_loader):,} | Total opt steps: {total_steps:,}\n")

#  Training loop — epoch 3 only 
global_step   = completed_steps   # continue global count from step ~4,078
best_val_loss = 1.7153            # ←  current best,  

print("=" * 55)
print("  RESUMING — EPOCH 3 ONLY")
print("=" * 55)

for epoch in range(RESUME_FROM, EPOCHS):
    model.train()
    optimizer.zero_grad()
    epoch_loss, nan_count, valid_count = 0.0, 0, 0
    epoch_start = time.time()

    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].cuda()
        attention_mask = batch["attention_mask"].cuda()
        labels         = batch["labels"].cuda()

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            loss = outputs.loss / ACCUM_STEPS

        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            optimizer.zero_grad()
            scaler.update()
            del outputs, loss, input_ids, attention_mask, labels
            torch.cuda.empty_cache()
            continue

        scaler.scale(loss).backward()
        epoch_loss  += loss.item() * ACCUM_STEPS
        valid_count += 1
        del outputs, loss, input_ids, attention_mask, labels

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            torch.cuda.empty_cache()

            if global_step % 50 == 0:
                avg  = epoch_loss / max(valid_count, 1)
                mins = (time.time() - epoch_start) / 60
                free, _ = torch.cuda.mem_get_info()
                print(f"  Ep {epoch+1} | Step {global_step:,} | "
                      f"Loss {avg:.4f} | NaN {nan_count} | "
                      f"GPU free {free/1e9:.1f} GB | {mins:.1f} min")

    #  Validation 
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                out = model(input_ids=batch["input_ids"].cuda(),
                            attention_mask=batch["attention_mask"].cuda(),
                            labels=batch["labels"].cuda())
            if not torch.isnan(out.loss) and not torch.isinf(out.loss):
                val_loss  += out.loss.item()
                val_steps += 1
            del out
        torch.cuda.empty_cache()

    avg_val   = val_loss / max(val_steps, 1)
    avg_train = epoch_loss / max(valid_count, 1)
    ep_mins   = (time.time() - epoch_start) / 60

    print(f"\n   Epoch {epoch+1} | Train {avg_train:.4f} | "
          f"Val {avg_val:.4f} | NaN skipped {nan_count} | {ep_mins:.1f} min\n")

    if avg_val < best_val_loss and val_steps > 0:
        best_val_loss = avg_val
        model.save_pretrained(f"{SAVE_DIR}/best_model")
        tokenizer.save_pretrained(f"{SAVE_DIR}/best_model")
        print(f"   Saved best model (val={best_val_loss:.4f})\n")

print("=" * 55)
print(f"  EPOCH 3 COMPLETE — best val loss: {best_val_loss:.4f}")
print("=" * 55)


Clean rows: 137,418
Train: 130,547  |  Val: 6,871


`torch_dtype` is deprecated! Use `dtype` instead!


Tokenizer loaded 


Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Model loaded  — GPU free: 13.05 GB


/tmp/ipykernel_1249257/4254344323.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...



Sanity loss: 1.6661  ← must not be nan
Post-sanity GPU free: 13.00 GB
 Sanity passed — resuming from epoch 3

Scheduler fast-forwarded 4078 steps ✅
Train batches: 65,274 | Total opt steps: 6,117

  RESUMING — EPOCH 3 ONLY
  Ep 3 | Step 4,100 | Loss 1.7700 | NaN 0 | GPU free 8.0 GB | 2.4 min
  Ep 3 | Step 4,150 | Loss 1.7605 | NaN 0 | GPU free 8.0 GB | 7.8 min
  Ep 3 | Step 4,200 | Loss 1.7602 | NaN 0 | GPU free 8.0 GB | 13.2 min
  Ep 3 | Step 4,250 | Loss 1.7532 | NaN 0 | GPU free 8.0 GB | 18.6 min
  Ep 3 | Step 4,300 | Loss 1.7516 | NaN 0 | GPU free 8.0 GB | 24.0 min
  Ep 3 | Step 4,350 | Loss 1.7457 | NaN 0 | GPU free 8.0 GB | 29.3 min
  Ep 3 | Step 4,400 | Loss 1.7454 | NaN 0 | GPU free 8.0 GB | 34.6 min
  Ep 3 | Step 4,450 | Loss 1.7417 | NaN 0 | GPU free 8.0 GB | 40.0 min
  Ep 3 | Step 4,500 | Loss 1.7387 | NaN 0 | GPU free 8.0 GB | 45.3 min
  Ep 3 | Step 4,550 | Loss 1.7385 | NaN 0 | GPU free 8.0 GB | 50.7 min
  Ep 3 | Step 4,600 | Loss 1.7396 | NaN 0 | GPU free 8.0 GB | 56.0 mi

In [2]:
import shutil, os
from IPython.display import FileLink

shutil.make_archive("/tmp/best_model_backup", "zip", f"{SAVE_DIR}/best_model")
size = os.path.getsize("/tmp/best_model_backup.zip") / 1e6
print(f" Zipped — {size:.1f} MB")
FileLink("/tmp/best_model_backup.zip")


 Zipped — 2290.9 MB


/tmp/best_model_backup.zip